Compute evaporative fraction
--
From daily LH and SH in regional simulations

In [1]:
cd ~/Pythons

/home/users/guicha/Pythons


In [2]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt

from matplotlib.colors import BoundaryNorm

from config import KSCALEOUTDIR
from KSCALE.read_data.read_data_catalog import load_data_regional
from KSCALE.p_config import regions

/home/users/guicha/.conda/envs/hk25/lib/python3.12/site-packages/pyproj/network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


In [3]:
### FUNC ###

def compute_ef(lhdata, shdata):
    """Compute evaporative fraction"""
    ef = lhdata / (lhdata + shdata)
    ef = ef.where((lhdata <= 0.) & (shdata <= 0.))  # compute EF only with both fluxes towards the atmosphere
    out = ef.where((ef >= 0.) & (ef <= 1.))

    return out

In [8]:
region='CTC'
resolution='km4p4'
physics='RAL3P3'
driving='GAL9'
reso_nest = 'n1280'
zoom = 7
year = 2020

ts = '1D'  # resampling timescale

year = 2020
months = np.arange(1, 12+1, 1)

conf = 'um_' + region + '_' + resolution + '_' + physics + '_' + reso_nest + '_' + driving + '_nest'

In [5]:
if region == 'CTC':
    lat_range = regions['Africa'][0]
    lon_range = regions['Africa'][1]
else:
    lat_range = regions[region][0]
    lon_range = regions[region][1]

lat_min = lat_range[0]
lat_max = lat_range[1]
lon_min = lon_range[0]
lon_max = lon_range[1]

In [6]:
#~ Outdir

if not os.path.isdir(KSCALEOUTDIR + '/ef'):
    os.mkdir(KSCALEOUTDIR + '/ef')
outdir = KSCALEOUTDIR + '/ef'

if not os.path.isdir(outdir + '/' + conf):
    os.mkdir(outdir + '/' + conf)
outdir = outdir + '/' + conf

if not os.path.isdir(outdir + '/z' + str(zoom)):
    os.mkdir(outdir + '/z' + str(zoom))
outdir = outdir + '/z' + str(zoom)

if not os.path.isdir(outdir + '/lat={0},{1}_lon={2},{3}'.format(lat_min, lat_max, lon_min, lon_max)):
    os.mkdir(outdir + '/lat={0},{1}_lon={2},{3}'.format(lat_min, lat_max, lon_min, lon_max))
outdir = outdir + '/lat={0},{1}_lon={2},{3}'.format(lat_min, lat_max, lon_min, lon_max)

outdir

'/gws/nopw/j04/kscale/USERS/guicha/outputs/data/ef/um_CTC_km4p4_RAL3P3_n1280_GAL9_nest/z7/lat=-35.0,25.0_lon=-20.0,52.0'

In [9]:
#~ Get data

data_lh = load_data_regional(region=region, resolution=resolution, physics=physics, driving=driving, reso_nest=reso_nest,
                                   zoom=zoom, variable='hflsd', lat_range=lat_range, lon_range=lon_range, year=year, timescale=ts)
data_sh = load_data_regional(region=region, resolution=resolution, physics=physics, driving=driving, reso_nest=reso_nest,
                                   zoom=zoom, variable='hfssd', lat_range=lat_range, lon_range=lon_range, year=year, timescale=ts)

In [10]:
ef = compute_ef(data_lh, data_sh)

In [12]:
#~ Save

outfile = outdir + '/' + str(year) + '_' + ts + '.nc'
ef.to_netcdf(outfile)
outfile

'/gws/nopw/j04/kscale/USERS/guicha/outputs/data/ef/um_CTC_km4p4_RAL3P3_n1280_GAL9_nest/z7/lat=-35.0,25.0_lon=-20.0,52.0/2020_1D.nc'